# Common inputs

## Libraries

In [ ]:
import gc
import importlib
import math
import os
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import matplotlib.colors as mcolors
import time

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)
pd.options.display.float_format = "{:,.2f}".format
%matplotlib inline

## imports functions

In [ ]:
CLEAN = True

INPUT_PATH = "../data"
OUTPUT_PATH = "../output/crm_results/results_files"
FIGURE_PATH = "../output/crm_results/figures"

import climate_risk_metric_functions as crm_functions
import default_assumptions
import fuel as fuel

# Reload modules if needed
import load_data as load
import policy as policy
import technology as technology
import utility_functions as utility_functions

importlib.reload(default_assumptions)
importlib.reload(load)
importlib.reload(utility_functions)
importlib.reload(fuel)
importlib.reload(technology)
importlib.reload(policy)
importlib.reload(crm_functions)

from climate_risk_metric_functions import (
    compute_ship_option_values,
    generate_interior_lhs_scenarios,
    process_operating_cost_batch,
    annual_transport_work,
)
from default_assumptions import (
    CARBON_CAPTURE_RATE,
    FUEL_DESIGNS,
    OPERATING_FUELS,
    PILOT_FUELS,
    REG_REG_CLASSES,
    WIND_ENERGY_SAVINGS,
    WIND_TECHNO_LIST,
)
from utility_functions import (
    clean_labels_for_plotting,
    clean_results_for_plotting,
    reduce_size_df,
)

# Variable-by-variable split of `scenario_variables` columns into genuinely
# categorical (methodology-selecting) vs. continuous (magnitude-with-a-low/high-bound)
# treatment for scenario generation. Names match the columns of
# `scenario_variables`/`scenarios` as built in `climate_risk_metric_v2.ipynb`
# (pre-rename -- the "RU prices"/"GFI trajectories" renames to "ru_prices"/"gfi"
# happen later, downstream of scenario construction).
DISCRETE_VARS = [
    "SU",
]
CONT_VARS = [
    "GFI trajectories", 
    "RU prices", 
    "Fuel reward e-fuels", 
    "e-fuels available after",
    "Fuel price bio", 
    "Fuel price e-fuels", 
    "Fuel price gas",    
    "Fuel reward bio", 
    "Fuel reward gas",
    "Carbon intensity bio", 
    "Carbon intensity e-fuels", 
    "Carbon intensity gas",
]

REWARD_DELAY = 1
DISCOUNT = 0.08

# Prepare operating costs
Fuel + compliance costs in a range of regulatory, technology and market scenarios

## Load files

In [ ]:
N_INTERIOR = 2500 # not used if distribution = "corners"
# Shape of the interior LHS blend weight (applies identically to every CONT_VARS
# variable): "uniform" is the central case (every point between a variable's low
# and high level equally likely); "triangular" is the alternative sensitivity case
# (symmetric triangular density peaked at the midpoint). #corners means all possible combinations
# Passed straight into
# generate_interior_lhs_scenarios, and should be reflected in any exported file name
# for this scenario set so uniform/triangular runs don't overwrite one another.
distribution = "uniform" # triangular, uniform, corners
policy_scenario = "baseline" # Choose between: baseline, nzf_and_reward, imo_strat

# scenarios
scenario_variables = pd.read_excel(
    f"{INPUT_PATH}/scenario_variables_crm.xlsx",
    sheet_name=f"scenario_input_{policy_scenario}",
    nrows=3,
).set_index("Unnamed: 0")
scenario_mapping = pd.read_excel(
    f"{INPUT_PATH}/scenario_variables_crm.xlsx",
    sheet_name="mapping",
)
ccs_rate = CARBON_CAPTURE_RATE

# Other assumptions
model_excel_db = "data/input/Model_Database.xlsx"

# Fuels
pilot_fuel_in = load.load_pilot_fuel_input(model_excel_db)
engine_fuels_dict, drop_in_fuels_in = load.load_engine_fuels_dict(
    model_excel_db, OPERATING_FUELS.fuel.unique()
)
# filter in only the fuels and design fuels in the CRM
drop_in_fuels_in = drop_in_fuels_in.loc[
    drop_in_fuels_in.fuel.isin(OPERATING_FUELS.fuel),
    [x for x in drop_in_fuels_in.columns if x in FUEL_DESIGNS] + ["fuel"],
]
energy_density = load.load_energy_density_table_alt(model_excel_db).reset_index()
energy_density = energy_density[
    energy_density.fuel.isin(OPERATING_FUELS.fuel)
]

wtt_ef, ttw_ef, fuel_engine_mapping = load.load_carbon_intensity(model_excel_db)
fuel_engine_mapping = fuel_engine_mapping[
    fuel_engine_mapping.fuel.isin(OPERATING_FUELS.fuel)
]
fuel_prices = load.load_fuel_price_data_usd_tonne_lshfoe(
    model_excel_db
)  # in $/tonne LSFOe

engines_design_mapping_in = (
    load.load_engine_design_mapping(model_excel_db)
    .drop_duplicates()
    .reset_index(drop=True)
)

# policy
eu_policy = "EU policy"
df_imo_targets_data, df_fueleu_targets_data = load.load_gfi_targets_data(model_excel_db)
sub = df_fueleu_targets_data[
    df_fueleu_targets_data.scenario == eu_policy
]
assert len(sub)>0
gfi = df_imo_targets_data.merge(
    sub[["year", "FuelEU"]],
    on="year",
    how="left",
)
assert len(gfi) == len(df_imo_targets_data)
# replace the 100 by 94, so that the uniform distribution makes sense
# this is later replaced by 100, so that the policy.py functions work (100 = no policy scenario)
gfi = gfi.set_index(["year",'scenario']).replace(100,94).reset_index()

df_ru_price_data = load.load_ru_price_data(model_excel_db)
ru_prices = df_ru_price_data.merge(
    sub[["year", "fueleu_penalty", "ets_price"]],
    on="year",
    how="left",
)
assert len(ru_prices) == len(df_ru_price_data)

reward = load.load_imo_nzf_reward_data(model_excel_db)
reward = reward[['scenario','year','value']]
for fuel in OPERATING_FUELS.fuel:
    if "e-" in fuel:
        reward[fuel] = reward["value"] * 41.2
    else:
        reward[fuel] = 0
reward = reward.drop(columns=["value"])  # in $/tonne LSFOe

## Prepare the skeleton of op_cost
This is specific to the climate risk metrics

### Set up the operating scenarios 

In [ ]:
# --- scenario definition---

# check that the scenario input files are properly set up
i = 0
CONT_VARS_FOR_INPUT = CONT_VARS.copy()
DISCRETE_VARS_FOR_INPUT = DISCRETE_VARS.copy()
for ind, row in scenario_mapping.iterrows():
    assert row['variable'] in CONT_VARS_FOR_INPUT + DISCRETE_VARS_FOR_INPUT, f"{row['variable']} not in the list of CONT and DISC VARS"
    if row["variable"] not in ["SU"]:
        list_1 = scenario_variables.loc[:, row["variable"]].to_list()
        list_1 = [x for x in list_1 if pd.notna(x)]
        assert len(list_1) > 0, f"Scenarios need to be defined for {row['variable']}"

        if row['variable'] in CONT_VARS_FOR_INPUT:
            # if a variable is continous in nature but only one scenario was given
            # or if all variables should be treated as discrete,
            # recast it as discrete
            if len(list_1) == 1 or distribution == "corners":
                CONT_VARS_FOR_INPUT.remove(row["variable"])
                DISCRETE_VARS_FOR_INPUT.append(row["variable"])

            # check that continous variables have the right number of scenarios
            # this is only applicable in the case where we apply a continuous distribution
            # for a corner distribution, all variables are de facto discrete
            if distribution != "corners":
                assert len(list_1) <=2, f"Too many scenarios defined for continuous variable {row['variable']}"

        # check that the model database contains the scenario
        list_2 = locals()[row["file"]]["scenario"].unique()
        missing = [x for x in list_1 if x not in list_2]
        if len(missing) > 0:
            i = i + 1
            print(row)
            print(missing)
        if row["breakdown_by_fuel"] == True:
            fuels = locals()[row["file"]].columns
            missing = [x for x in OPERATING_FUELS["fuel"] if x not in fuels]
            if len(missing) > 0:
                i = i + 1
                print(row)
                print(missing)

# interior Sliced Latin Hypercube Design scenarios, sampled continuously over
# CONT_VARS. The discrete columns passed in (DISCRETE_VARS)
# are automatically classified by generate_interior_lhs_scenarios itself, from
# scenario_variables: any column with more than one distinct defined level becomes a
# *slicing* variable (the N draws are partitioned into one independently-sampled
# batch per combination of slicing variables' levels), while any column with only a
# single defined level is simply held fixed throughout interior sampling. Today only
# "SU" has 2 levels, so it is the only variable that ends up slicing -- but this is
# derived from the data, not hardcoded, so it adapts automatically if more
# DISCRETE_VARS columns get a second level in the future.
# Interior scenarios are reported separately from the corner scenarios above (not
# pooled into one weighted set).
# `distribution` ("uniform" central case / "triangular" alternative case, set above
# in the data-loading cell) shapes the blend weight identically across all CONT_VARS
# variables -- see generate_interior_lhs_scenarios's docstring for details.
scenarios, updated_reference_tables = generate_interior_lhs_scenarios(
    N_INTERIOR,
    scenario_variables=scenario_variables,
    scenario_mapping=scenario_mapping,
    reference_tables={
        "gfi": gfi,
        "ru_prices": ru_prices,
        "reward": reward,
        "fuel_prices": fuel_prices,
        "wtt_ef": wtt_ef,
    },
    cont_vars=CONT_VARS_FOR_INPUT,
    discrete_vars=DISCRETE_VARS_FOR_INPUT,
    distribution=distribution,
    seed=0,
)
# the interior draws' synthetic scenario labels only resolve correctly downstream once
# these updated reference tables (with the newly-materialized blended rows) replace
# the originals that the rest of the notebook looks up by name
gfi = updated_reference_tables["gfi"]
gfi = gfi.set_index(["year",'scenario']).replace(94,100).reset_index()
ru_prices = updated_reference_tables["ru_prices"]
reward = updated_reference_tables["reward"]
fuel_prices = updated_reference_tables["fuel_prices"]
wtt_ef = updated_reference_tables["wtt_ef"]

print(f"Number of scenarios: {len(scenarios)}")

scenarios["scenario_id"] = scenarios.index
print(f"n scenarios: {len(scenarios["scenario_id"].unique())}")

scenarios.sample(3)

In [ ]:
# we drop the EU policy if the IMO policy's stringency is close enough to the IMO NZF as per Apr 2025
# also, the reward threshold is not subject to uncertainty - so we leave it the standard level if there is any IMO policy
tol = 0.9 # how close the policy needs to be to the NZF

nzf_gfi = gfi[gfi.scenario == 'IMO NZF'][['year','base_compliance_target','direct_compliance_target','znz']].rename(
    columns = {'base_compliance_target':'nzf_base_compliance_target','direct_compliance_target':'nzf_direct_compliance_target','znz':'nzf_znz'}
)
check_len = len(gfi)
gfi = gfi.merge(
    nzf_gfi,
    on = 'year',
    how = 'left'
)
assert len(gfi) == check_len, f"Mismatch in the lengths after merge: {check_len} vs {len(gfi)}"
assert len(gfi[gfi.nzf_base_compliance_target.isna()]) == 0, f"Nan after merge: {gfi[gfi.nzf_base_compliance_target.isna()].sample()}"

mask_drop_eu_pol = gfi[
    (gfi.base_compliance_target * tol<=gfi.nzf_base_compliance_target)
    &(gfi.direct_compliance_target* tol<=gfi.nzf_direct_compliance_target)
].index
gfi.loc[mask_drop_eu_pol, 'FuelEU'] = 100
# keep a mask of the stringency flag for future filtering
gfi['stringency_nzf_flag'] = False
gfi.loc[mask_drop_eu_pol, 'stringency_nzf_flag'] = True

# put the ZNZ threshold back to its original level
gfi['znz'] = 0
mask_imo_pol = gfi[
    (gfi.base_compliance_target > 0)
    |(gfi.direct_compliance_target > 0)
].index
gfi.loc[mask_imo_pol, 'znz'] = gfi.loc[mask_imo_pol, 'nzf_znz']
gfi.drop(columns = {'nzf_base_compliance_target','nzf_direct_compliance_target','nzf_znz'}, inplace = True)

In [ ]:
# create a subset for each year and each operating fuel and only with operating-related inputs

scenarios = scenarios.copy()
for col in [
    "RU prices",
    "GFI trajectories",
    "Fuel reward bio",
    "Fuel reward e-fuels",
    "Fuel reward gas",
    "Fuel price bio",
    "Fuel price e-fuels",
    "Fuel price gas",
    "Carbon intensity bio",
    "Carbon intensity e-fuels",
    "Carbon intensity gas",
    "SU",
    "scenario_id",
]:
    scenarios[col] = scenarios[col].astype("category")

year_fuel_reg_index = pd.MultiIndex.from_product(
    [
        range(2025, 2051, 1),
        OPERATING_FUELS["fuel"].tolist(),
        REG_REG_CLASSES,
    ],
    names=["year", "fuel", "reg_reg_class"],
).to_frame(index=False)
year_fuel_reg_index["year"] = year_fuel_reg_index["year"].astype("int16")
year_fuel_reg_index["fuel"] = year_fuel_reg_index["fuel"].astype("category")
year_fuel_reg_index["reg_reg_class"] = year_fuel_reg_index["reg_reg_class"].astype(
    "category"
)

operating_scenarios = year_fuel_reg_index.merge(scenarios, how="cross")
operating_scenarios = operating_scenarios.reset_index(drop=True)
assert len(operating_scenarios) == len(OPERATING_FUELS["fuel"]) * len(scenarios) * (
    2050 - 2025 + 1
) * len(REG_REG_CLASSES)

# `Fuel reward bio`/`Fuel reward e-fuels`/`Fuel reward gas` (and their fuel_price/carbon
# intensity equivalents) are each categorical with their own distinct categories. 
# Initialize `reward`/`fuel_prices`/`wtt_ef` as plain object columns first,
# so assigning from a *different* one of these per fuel_group below doesn't hit pandas'
# "identical categories" check on a column that became categorical from the 1st assignment.
for col in ["reward", "fuel_prices", "wtt_ef"]:
    operating_scenarios[col] = pd.Series(
        pd.NA, index=operating_scenarios.index, dtype="object"
    )

for fuel_group in OPERATING_FUELS.fuel_group.unique():
    ind = operating_scenarios[
        operating_scenarios.fuel.isin(
            OPERATING_FUELS[OPERATING_FUELS.fuel_group == fuel_group].fuel
        )
    ].index
    operating_scenarios.loc[ind, "reward"] = operating_scenarios.loc[
        ind, f"Fuel reward {fuel_group}"
    ]
    operating_scenarios.loc[ind, "fuel_prices"] = operating_scenarios.loc[
        ind, f"Fuel price {fuel_group}"
    ]
    operating_scenarios.loc[ind, "wtt_ef"] = operating_scenarios.loc[
        ind, f"Carbon intensity {fuel_group}"
    ]
    operating_scenarios.loc[ind, "fuel_group"] = fuel_group
operating_scenarios = operating_scenarios.rename(
    columns={"RU prices": "ru_prices", "GFI trajectories": "gfi"}
)

# reward/fuel_prices/wtt_ef are only assembled just above (from the already-categorical
# per-fuel-group columns), so they can only be cast to category here, right after they're
# populated
for col in ["reward", "fuel_prices", "wtt_ef"]:
    operating_scenarios[col] = operating_scenarios[col].astype("category")

print(f"n scenarios: {len(operating_scenarios["scenario_id"].unique())}")

operating_scenarios.sample(3)


In [ ]:
# --- input loading---

# Merge the relevant operating inputs to reach year * fuel * scenario
operating_cost_pre_batch = operating_scenarios.copy()
check_len = len(operating_scenarios)

for var in [
    "ru_prices",
    "gfi",
    "reward",
    "wtt_ef",
    "fuel_prices",
]:
    to_merge = locals()[var]
    to_merge["year"] = to_merge["year"].astype(int)

    if (
        scenario_mapping[scenario_mapping.file == var]["breakdown_by_fuel"].iloc[0]
        == True
    ):
        # Only the actual fuel columns should be melted into fuel/value pairs.
        fuel_cols = [c for c in to_merge.columns if c in OPERATING_FUELS["fuel"].tolist()]
        extra_cols = [
            c for c in to_merge.columns if c not in fuel_cols + ["scenario", "year"] and c != "uranium"
        ]
        print(extra_cols)

        to_merge_melt = to_merge[["scenario", "year"] + extra_cols + fuel_cols].melt(
            id_vars=["scenario", "year"] + extra_cols, value_name=f"{var}_value", var_name="fuel"
        )
        operating_cost_pre_batch = operating_cost_pre_batch.merge(
            to_merge_melt.rename(columns={"scenario": var}),
            on=["year", var, "fuel"],
            how="left",
        )

    else:
        operating_cost_pre_batch = operating_cost_pre_batch.merge(
            to_merge.rename(columns={"scenario": var}),
            on=["year", var],
            how="left",
        )

    assert check_len == len(operating_cost_pre_batch), f"{var}, check_len: {check_len} vs len(operating_cost_pre_batch):{len(operating_cost_pre_batch)}"
assert not np.any(
    operating_cost_pre_batch[
        [
            "ru1",
            "ru2",
            "ets_price",
            "fueleu_penalty",
            "base_compliance_target",
            "direct_compliance_target",
            "znz",
            "FuelEU",
            "reward_value",
            "fuel_prices_value",
            "wtt_ef_value",
        ]
    ].isna()
), "NaNs found in some columns"

# --- TTW emission factors ---
check_len = len(operating_cost_pre_batch)
operating_cost_pre_batch = operating_cost_pre_batch.merge(fuel_engine_mapping, on="fuel", how="left")
assert len(operating_cost_pre_batch) == (
    check_len * len(fuel_engine_mapping) / len(fuel_engine_mapping["fuel"].unique())
)
check_len = len(operating_cost_pre_batch)

operating_cost_pre_batch = operating_cost_pre_batch.merge(ttw_ef, on=["fuel", "engine_type"], how="left")
assert check_len == len(operating_cost_pre_batch)

assert not np.any(operating_cost_pre_batch["ttw_gco2e_p_mj"].isna()), "NaNs found in ttw_gco2e_p_mj"
operating_cost_pre_batch["reward_delay"] = REWARD_DELAY
operating_cost_pre_batch["wacc"] = DISCOUNT

# ============= MANUAL AMENDMENTS OF SCENARIOS =============
# Remove e-fuels for years without fuel infrastructure
operating_cost_pre_batch = operating_cost_pre_batch[
    (operating_cost_pre_batch.fuel_group != "e-fuels")
    | (operating_cost_pre_batch.year >= operating_cost_pre_batch['e-fuels available after'])
].reset_index(drop=True)

# Remove FuelEU if the NZF is sufficiently stringent
mask = operating_cost_pre_batch[
    (operating_cost_pre_batch.stringency_nzf_flag)
].index
operating_cost_pre_batch.loc[mask, ['fueleu_penalty', 'ets_price']] = 0
operating_cost_pre_batch.loc[mask, 'FuelEU'] = 100


n_scenarios = len(operating_cost_pre_batch["scenario_id"].unique())
print(f"n scenarios: {n_scenarios}")

operating_cost_pre_batch.sample(3)

### Plotchecking

In [ ]:
vars_to_plot = [
    "reward",
    "fuel_prices",
    "wtt_ef",
]

subset = operating_cost_pre_batch[
    (
        (
            (operating_cost_pre_batch.fuel == "lng")
            & (operating_cost_pre_batch.engine_type == "LNG Diesel 2024+")
        )
        | (operating_cost_pre_batch.fuel != "lng")
    )
    & (operating_cost_pre_batch.reg_reg_class == "noneu_to_noneu")
]
subset = clean_labels_for_plotting(subset, ["fuel"])
cols = subset.fuel.unique()
ncols = 2
nrows = math.ceil(len(cols) / ncols)

for var in vars_to_plot:
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=(10, 4 * nrows), sharey=True
    )
    axes = np.array(axes).flatten()

    for j, fuel in enumerate(cols):
        ax = axes[j]

        pivoted = (
            subset[subset["fuel"] == fuel]
            .groupby(["year", var])[f"{var}_value"]
            .mean()
            .unstack()
        )

        if not pivoted.empty:  # better than sum>0
            pivoted.plot(ax=ax)
            ax.set_title(f"{var} – {fuel}")
            ax.set_xlabel("")
            ax.set_ylabel(scenario_mapping[scenario_mapping.file == var].unit.iloc[0])

            if var == "fuel_prices":
                ax.set_ylim(0, 4000)
            elif var == "ru_prices":
                ax.set_ylim(0, 800)
            elif var == "wtt_ef":
                ax.set_ylim(0, 80)
            elif var == "reward":
                ax.set_ylim(0, 2000)
            else:
                ax.set_ylim(bottom=0)

            ax.legend_.remove() if ax.legend_ else None

    # hide unused axes
    for k in range(len(cols), len(axes)):
        fig.delaxes(axes[k])

    fig.tight_layout()
    plt.show()

In [ ]:
x_var = "base_compliance_target"
y_var = "fuel_prices_value"

to_plot = operating_cost_pre_batch[
    (operating_cost_pre_batch.fuel == "lng")
    &(operating_cost_pre_batch.year == 2035)
]
plt.scatter(x = to_plot[x_var], y = to_plot[y_var])
plt.xlabel(x_var)
plt.ylabel(y_var)
plt.show()

In [ ]:
# Plot uncertainty ranges for selected fuels in $/t LSFO2

# fuels_to_plot = ['bio-LNG','bio methanol/bio-ethanol', 'bio-MGO', ]
fuels_to_plot = ['e-MGO', 'e-LNG','e-methanol', 'e-ammonia']
# fuels_to_plot = ['blue ammonia','LNG','fuel oil']

# Filter data for selected fuels and engine type
subset_data = operating_cost_pre_batch[
    # (operating_cost_pre_batch.fuel.isin(fuels_to_plot))
    # & (operating_cost_pre_batch.engine_type == "All")
    (operating_cost_pre_batch.reg_reg_class == "noneu_to_noneu")
].copy()

subset_data = clean_labels_for_plotting(subset_data, ["fuel"])

# For each fuel and year, calculate min and max across all scenarios
uncertainty_ranges = subset_data.groupby(['fuel', 'year'])['fuel_prices_value'].agg([
    ('low', 'min'),
    ('high', 'max'),
]).reset_index()
plt.close()

# Create figure with shaded ranges
fig, ax = plt.subplots(figsize=(8, 6))

# Define colors for each fuel
colors = [ '#ff7f0e',"#646464",'#1f77b4','#2ca02c']

i = 0
for fuel in fuels_to_plot:
    fuel_data = uncertainty_ranges[uncertainty_ranges.fuel == fuel].sort_values('year')
    
    # Plot the shaded range
    ax.fill_between(
        fuel_data['year'],
        fuel_data['low'],
        fuel_data['high'],
        alpha=0.3,
        color=colors[i],
        label=fuel,
        hatch='///' if fuel == 'bio-mgo' else None,
    )
    i=i+1

ax.set_xlabel("")
ax.set_ylabel('Fuel Price ($/t LSFOe)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(2024.5, 2050.5)

plt.tight_layout()
plt.show()

In [ ]:
# Plot uncertainty ranges for selected fuels in $/t LSFO2

# fuels_to_plot = ['bio-LNG','bio methanol/bio-ethanol', 'bio-MGO', ]
fuels_to_plot = ['e-MGO', 'e-LNG','e-methanol', 'e-ammonia']
# fuels_to_plot = ['blue ammonia','LNG','fuel oil']

# Filter data for selected fuels and engine type
subset_data = operating_cost_pre_batch[
    (operating_cost_pre_batch.reg_reg_class == "noneu_to_noneu")
].copy()

subset_data = clean_labels_for_plotting(subset_data, ["fuel"])

# For each fuel and year, calculate min and max across all scenarios
uncertainty_ranges = subset_data.groupby(['fuel', 'year'])['fuel_prices_value'].agg([
    ('low', 'min'),
    ('high', 'max'),
]).reset_index()
plt.close()

# Create figure with shaded ranges
fig, ax = plt.subplots(figsize=(8, 6))

# Define colors for each fuel
colors = [ '#ff7f0e',"#646464",'#1f77b4','#2ca02c']

i = 0
for fuel in fuels_to_plot:
    fuel_data = uncertainty_ranges[uncertainty_ranges.fuel == fuel].sort_values('year')
    
    # Plot the shaded range
    ax.fill_between(
        fuel_data['year'],
        fuel_data['low'],
        fuel_data['high'],
        alpha=0.3,
        color=colors[i],
        label=fuel,
        hatch='///' if fuel == 'bio-mgo' else None,
    )
    i=i+1

ax.set_xlabel("")
ax.set_ylabel('Fuel Price ($/t LSFOe)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(2024.5, 2050.5)

plt.tight_layout()
plt.show()

In [ ]:
if CLEAN:
    del subset_data
    del to_plot
    del subset
    gc.collect()

### Prepare base data for batch processing

In [ ]:
# Prepare base (pre-expansion) data for batch processing
# Wind/CCS expansion will happen inside process_operating_cost_batch() for each batch

# Filter engines_design_mapping to remove "other" and "nuclear" designs
engines_design_mapping_filtered = engines_design_mapping_in[
    ~(engines_design_mapping_in.fuel_design.str.contains("other"))
    & ~(engines_design_mapping_in.fuel_design.str.contains("nuclear"))
].copy()

print(f"Base data prepared:")
print(f"  operating_cost_pre_batch: {len(operating_cost_pre_batch):,} rows (pre-expansion)")
print(f"  drop_in_fuels_in: {len(drop_in_fuels_in):,} rows")
print(f"  pilot_fuel_in: {len(pilot_fuel_in):,} rows")
print(f"  engines_design_mapping_filtered: {len(engines_design_mapping_filtered):,} rows")
print(f"  fuel_engine_mapping: {len(fuel_engine_mapping):,} rows")
print(f"\nWind/CCS expansion and manual amendments will happen inside each batch")
operating_cost_pre_batch.sample(3)

In [ ]:
# ============================================================================
# VALIDATIONS
# ============================================================================
def check_match(name_a, set_a, name_b, set_b):
    """Prints a nice consistency check comparing two sets."""
    if set_a == set_b:
        print(f"✔ {name_a} matches {name_b}.")
    else:
        print(f"❌ MISMATCH between {name_a} and {name_b}!")
        missing = set_a - set_b
        extra = set_b - set_a
        if missing:
            print(f"   Missing in {name_b}: {missing}")
        if extra:
            print(f"   Unexpected extras in {name_b}: {extra}")
    print("")

check_match(
    "drop_in_fuels_in.fuel",
    set(drop_in_fuels_in["fuel"].unique()),
    "operating_cost_pre_batch.fuel",
    set(operating_cost_pre_batch["fuel"].unique()),
)

check_match(
    "drop_in_fuels_in.fuel",
    set(drop_in_fuels_in["fuel"].unique()),
    "fuel_engine_mapping.fuel",
    set(fuel_engine_mapping["fuel"].unique()),
)

check_match(
    "drop_in_fuels_in.fuel_design",
    set(drop_in_fuels_in.columns),
    "engines_design_mapping_filtered.fuel_design",
    set(engines_design_mapping_filtered["fuel_design"].unique()),
)

## Calculate operating costs

In [ ]:
# Define policy compliance parameter columns (used in batching function)
policy_compliance_params_cols = [
    "year",
    "reward_scenario",
    "base_compliance_target",
    "direct_compliance_target",
    "reward_threshold_gco2_mj",
    "ru1_price",
    "ru2_price",
    "su_price",
    "reward_usd_tlsfoeq",
    "reg_reg_class",
    "FuelEU",
    "pooling_value",
    "FuelEU_penalty",
    "EU_ETS",
    "wacc",
    "reward_delay",
    "fuel",
    "wtw_gco2e_p_mj",
    "ttw_gco2e_p_mj",
    "eu_pooling_price",
]

# ============= BATCHED OPERATING COST CALCULATION =============
# Process scenarios in batches to manage memory
# Each batch: operating_cost_pre_batch (pre-expansion) -> wind/CCS expansion -> manual amendments -> full pipeline

scenario_ids = operating_cost_pre_batch['scenario_id'].unique()
assert len(scenario_ids) == n_scenarios, f"Inconsistent number of scenarios: {len(scenario_ids)} vs {n_scenarios}"
batch_size = 100  # Tune based on available RAM (100 scenarios ≈ ~3.3M rows ≈ ~650 MB)

print(f"Processing {len(scenario_ids)} scenarios in batches of {batch_size}...")

operating_cost_batches = []

for batch_start in tqdm(range(0, len(scenario_ids), batch_size), desc="Scenario batches"):
    batch_scenario_ids = scenario_ids[batch_start:batch_start + batch_size]
    
    # Filter to current batch (pre-expansion data)
    op_cost_batch = operating_cost_pre_batch[
        operating_cost_pre_batch['scenario_id'].isin(batch_scenario_ids)
    ].copy()
    
    print(f"\nBatch {batch_start//batch_size + 1}: {len(batch_scenario_ids)} scenarios, {len(op_cost_batch):,} rows (pre-expansion)")
    
    # Process batch through full pipeline (expansion happens inside function)
    operating_cost_batch, drop_in_fuels, engines_design_mapping = process_operating_cost_batch(
        op_cost_batch,
        drop_in_fuels_in=drop_in_fuels_in,
        pilot_fuel_in=pilot_fuel_in,
        engines_design_mapping_in=engines_design_mapping_filtered,
        fuel_engine_mapping_in=fuel_engine_mapping,
        WIND_TECHNO_LIST=WIND_TECHNO_LIST,
        WIND_ENERGY_SAVINGS=WIND_ENERGY_SAVINGS,
        energy_density=energy_density,
        PILOT_FUELS=PILOT_FUELS,
        policy_compliance_params_cols=policy_compliance_params_cols,
        ccs_rate=ccs_rate,
        CLEAN=CLEAN,
    )

    if CLEAN:
        cols_needed = [
            "scenario_id", "year", "engine_type", "reg_reg_class", "fuel", 
            "usd_p_GJ", "total_fuel_cost",
            "gfi",
        ]
    else:
        cols_needed = operating_cost_batch.columns
    
    operating_cost_batches.append(operating_cost_batch[cols_needed])
    
    # Explicit cleanup
    del op_cost_batch, operating_cost_batch
    gc.collect()
    
    print(f"Batch complete. Memory cleaned.")

# Combine all batches
print(f"\nCombining {len(operating_cost_batches)} batches...")
operating_cost = pd.concat(operating_cost_batches, ignore_index=True)

if CLEAN:
    del operating_cost_batches
    gc.collect()

print(f"Final operating_cost: {len(operating_cost):,} rows, {len(operating_cost['scenario_id'].unique())} scenarios")
operating_cost.sample(3)

## Cleaning and saving

### Scenarios streamlining

In [ ]:
# --- scenarios streamlining---

# to save some running time later on, we identify which scenarios lead to the exact same operating cost
# we remove those duplicates but keep a trace of the duplicates
# cols identifying the "features" of a scenario
# Columns that define the scenario's results across years/fuels/etc
if distribution == "corners":
    cols_check = ["year", "fuel", "engine_type", "reg_reg_class", "usd_p_GJ"]

    # 1. Row-level hash
    row_hashes = pd.util.hash_pandas_object(operating_cost[cols_check], index=False).values

    # 2. Compute scenario-level hashes via bincount
    operating_cost["scenario_id"] = operating_cost["scenario_id"].astype(int)
    scenario_ids = operating_cost["scenario_id"].values
    s_min = scenario_ids.min()
    temp_ids = scenario_ids - s_min  # ensure 0-based indexing
    n_scen = temp_ids.max() + 1

    scenario_hash_values = np.bincount(temp_ids, weights=row_hashes, minlength=n_scen)

    # 3. Map row → scenario hash
    scenario_hash_map = scenario_hash_values[temp_ids]

    # 4. Count duplicates per unique scenario hash
    unique_hashes, duplicate_counts = np.unique(scenario_hash_values, return_counts=True)
    
    # 5. Assign "# duplicates" to each row (use pandas Series.map for efficiency)
    hash_count_series = pd.Series(duplicate_counts, index=unique_hashes)
    operating_cost["# duplicates"] = pd.Series(scenario_hash_map).map(hash_count_series).astype(np.int32).values

    # 6. Keep only one scenario per unique hash
    unique_hash_indices = np.unique(scenario_hash_values, return_index=True)[1]
    unique_scenario_ids = unique_hash_indices + s_min
    
    # Use numpy isin for memory efficiency (faster than pandas .isin() on large arrays)
    mask = np.isin(scenario_ids, unique_scenario_ids)
    operating_cost_unique = operating_cost[mask].copy()

    # Clean up large arrays immediately
    del mask, row_hashes, scenario_ids, temp_ids, scenario_hash_values, scenario_hash_map, hash_count_series
    
    assert len(operating_cost_unique) <= len(operating_cost)
    if len(operating_cost_unique) != len(operating_cost):
        # Verify
        assert operating_cost_unique["# duplicates"].notna().all()
        assert (
            operating_cost_unique[["scenario_id", "# duplicates"]]
            .drop_duplicates()["# duplicates"]
            .sum()
            == n_scenarios
        )
    else:
        operating_cost_unique["# duplicates"] = 1
    print(f"Number of scenarios before {n_scenarios} and after {len(operating_cost_unique.scenario_id.unique())}")
    # Clean up before creating future years
    if CLEAN:
        del operating_cost
        gc.collect()

    # for years past 2050, just assume same as 2050
    # Extract 2050 data once
    year_2050_data = operating_cost_unique[operating_cost_unique.year == 2050].copy()
    
    # Instead of creating 20 copies, replicate year column and tile the data
    n_future_years = 20  # 2051-2070
    n_rows_2050 = len(year_2050_data)
    
    # Create future years more efficiently by tiling
    future_data = pd.concat([year_2050_data] * n_future_years, ignore_index=True)
    future_data["year"] = np.repeat(range(2051, 2071), n_rows_2050)
    
    # Concatenate once
    operating_cost_unique = pd.concat(
        [operating_cost_unique, future_data], ignore_index=True
    )
    
    if CLEAN:
        del future_data, year_2050_data
        gc.collect()

    operating_cost_unique = reduce_size_df(operating_cost_unique)
    operating_cost_unique["year"] = operating_cost_unique["year"].astype("int32")

    operating_cost = operating_cost_unique
    if CLEAN:
        del operating_cost_unique
        gc.collect()
else:
     operating_cost['# duplicates'] = 1

### Export files

In [ ]:
# export relevant files
drop_in_fuels.to_pickle(f"{OUTPUT_PATH}/drop_in_fuels.pkl")
engines_design_mapping.to_pickle(f"{OUTPUT_PATH}/engines_design_mapping.pkl")
operating_cost.to_pickle(
    f"{OUTPUT_PATH}/operating_cost_{n_scenarios}_ccs_{ccs_rate}_distr_{distribution}_{policy_scenario}.pkl.gz",
    compression="gzip",
)
# for reference
operating_cost_pre_batch.to_csv(f"{OUTPUT_PATH}/scenarios_{n_scenarios}_ccs_{ccs_rate}_distr_{distribution}_{policy_scenario}.pkl")

# Plotchecking

In [ ]:
# === INPUT ASSUMPTIONS VALIDATION PLOTS ===
# Verify input assumptions are correctly represented in operating_cost across scenarios

# Category 1: Grouped by scenario parameters (ru_prices, gfi)
# For each scenario parameter, plot how associated variables vary by year
category1 = {
    "ru_prices": ['ru1_price', 'ru2_price', 'ets_price', 'FuelEU_penalty'],
    "gfi": ['base_compliance_target', 'direct_compliance_target', 'znz', 'FuelEU'],
}

os.makedirs(FIGURE_PATH, exist_ok=True)

for var_sc, variables in category1.items():
    print(f"\nCreating plots for scenario parameter: {var_sc}")
    
    # Create figure with subplots (one per variable)
    ncols = 2
    nrows = (len(variables) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4*nrows))
    axes = axes.flatten()
    
    for idx, var in enumerate(variables):
        ax = axes[idx]
        try:
            pivoted = operating_cost[operating_cost.year<2051].groupby(['year', var_sc])[var].mean().unstack()
            pivoted.plot(ax=ax, legend=True)
            ax.set_title(f"{var} by year and {var_sc}")
            ax.set_xlabel("Year")
            ax.set_ylabel(f"Mean {var}")
            ax.grid(alpha=0.3)
        except Exception as e:
            ax.text(0.5, 0.5, f"Error plotting {var}:\n{str(e)}", 
                   ha='center', va='center', transform=ax.transAxes)
    
        ax.legend_.remove() if ax.legend_ else None
    # Hide unused subplots
    for idx in range(len(variables), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    filename = os.path.join(FIGURE_PATH, f"input_validation_{var_sc}.png")
    fig.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

# Category 2: Grouped by fuel and scenario parameter
# For each variable-scenario_param pair, show how variable varies by fuel and year
category2 = {
    "main_fuel_price_usd_tvslfoeq": "fuel_prices",
    "wtt_gco2e_p_mj": "wtt_ef",
}

# Get unique fuels
unique_fuels = [x for x in operating_cost['fuel'].dropna().unique() if "+" not in str(x) and "mix" not in str(x)]
fuels_to_plot = [f for f in unique_fuels if len(str(f)) < 50][:10]  # Limit to first 10 to avoid too many plots

for var1, var_sc in category2.items():
    print(f"\nCreating plots for {var1} (grouped by {var_sc})")
    
    # Create figure with subplots (one per fuel)
    ncols = 2
    nrows = (len(fuels_to_plot) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4*nrows))
    axes = axes.flatten()
    
    for idx, fuel in enumerate(fuels_to_plot):
        ax = axes[idx]
        try:
            df_fuel = operating_cost[(operating_cost['fuel'] == fuel) &(operating_cost.year<2051)]
            pivoted = df_fuel.groupby(['year', var_sc])[var1].mean().unstack()
            if len(pivoted) > 0:
                pivoted.plot(ax=ax, legend=True)
                ax.set_title(f"{fuel}")
                ax.set_xlabel("Year")
                ax.set_ylabel(f"{var1}")
                ax.grid(alpha=0.3)
            else:
                ax.text(0.5, 0.5, f"No data for fuel: {fuel}", 
                       ha='center', va='center', transform=ax.transAxes)
        except Exception as e:
            ax.text(0.5, 0.5, f"Error plotting {fuel}:\n{str(e)}", 
                   ha='center', va='center', transform=ax.transAxes)
        ax.legend_.remove() if ax.legend_ else None
    # Hide unused subplots
    for idx in range(len(fuels_to_plot), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()


In [ ]:
vars_to_plot = [
    "ru_prices",
    "gfi",
    "reward",
    "fuel_prices",
    "wtt_ef",
]

# Ensure output directory exists
os.makedirs(FIGURE_PATH, exist_ok=True)

# Loop over variables
for var in vars_to_plot:
    df = locals()[var].copy()
    df["year"] = df["year"].astype(int)
    df = df[df.scenario.isin(operating_cost[var])]

    # Collect the non-meta columns
    cols = [c for c in df.columns if c not in ["year", "scenario","stringency_nzf_flag"]]

    ncols = 2
    nrows = math.ceil(len(cols) / ncols)

    plt.close()
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=(10, 4 * nrows), sharey=True
    )
    axes = axes.flatten()

    # Loop over each column of this variable
    for j, col in enumerate(cols):
        ax = axes[j]
        pivoted = df.pivot(index="year", columns="scenario", values=col)
        if pivoted.sum().sum() > 0:
            pivoted.plot(ax=ax)
            ax.set_title(f"{var} – {col}")
            ax.set_xlabel("")
            ax.set_ylabel(scenario_mapping[scenario_mapping.file == var].unit.iloc[0])
            if var == "ru_prices":
                ax.set_ylim(0, 800)

            elif var == "wtt_ef":
                ax.set_ylim(0, 40)
            else:
                ax.set_ylim(0)
            ax.legend().remove()

    # Hide unused axes
    for k in range(len(cols), len(axes)):
        fig.delaxes(axes[k])


    filename = os.path.join(FIGURE_PATH, f"{var}.png")
    fig.savefig(filename, dpi=300)
    plt.show()

# Risk metric model run

## Load files

In [ ]:
n_scenarios = 2500

# Load operating costs
drop_in_fuels = pd.read_pickle(f"{OUTPUT_PATH}/drop_in_fuels.pkl")
engines_design_mapping = pd.read_pickle(f"{OUTPUT_PATH}/engines_design_mapping.pkl")
operating_cost = pd.DataFrame()
for ccs_rate in [CARBON_CAPTURE_RATE]:  
    for policy_scenario in [
        'baseline',
        # 'nzf_and_reward',
        # 'imo_strat',
        # "check",
    ]:
        for distribution in [
            # 'corners',
            'triangular',
            'uniform'
        ]:
            file_path = f"{OUTPUT_PATH}/operating_cost_{n_scenarios}_ccs_{ccs_rate}_distr_{distribution}_{policy_scenario}.pkl.gz"
            if os.path.exists(file_path):
                to_add = pd.read_pickle(file_path, compression="gzip")
                to_add["ccs_rate_sc"] = ccs_rate
                to_add["policy_scenario"] = policy_scenario
                to_add["distribution"] = distribution
                operating_cost = pd.concat((operating_cost, to_add), ignore_index=True)
                print(f"Added: {os.path.basename(file_path)}")

# rename to make th legends prettier
operating_cost["ccs_rate_sc"] = operating_cost["ccs_rate_sc"].replace(
    {0.2: "Low", 0.27: "Mid", 0.5: "High"}
).astype("category")
operating_cost["policy_scenario"] = operating_cost["policy_scenario"].replace(
    {'baseline': "No NZF is a possibility", "nzf_and_reward": "NZF and rewards adopted",'imo_strat':'IMO Strategy'}
).astype("category")

# MEMORY/CLEANING
# Recast any scenario-defining / low-cardinality columns that may have
# decayed to object (e.g. via .replace(), .merge(), or pickle round-trip)
cat_cols = [
    "fuel", "scenario_id", "year", "reg_reg_class", "engine_type",
    "ru_prices", "gfi", "SU", "ccs_rate_sc",
    "Fuel reward bio", "Fuel reward e-fuels", "Fuel reward gas",
    "Fuel price bio", "Fuel price e-fuels", "Fuel price gas",
    "Carbon intensity bio", "Carbon intensity e-fuels", "Carbon intensity gas",
    "policy_scenario","distribution"
]
for col in cat_cols:
    if col in operating_cost.columns and operating_cost[col].dtype == object:
        operating_cost[col] = operating_cost[col].astype("category")

cols_needed = [
    "scenario_id", "year", "engine_type", "reg_reg_class", "fuel", 
    "usd_p_GJ", "total_fuel_cost",
    "# duplicates", "ccs_rate_sc", "gfi", 
    "policy_scenario","distribution"
] 
operating_cost = operating_cost[cols_needed]

print(operating_cost.dtypes.value_counts())
print(f"Memory usage: {operating_cost.memory_usage(deep=True).sum() / 1e9:.2f} GB")

model_excel_db = "data/input/Model_Database.xlsx"

df_engine_efficiency_ref = load.load_engine_efficiency_data(model_excel_db)

_, _, fuel_engine_mapping = load.load_carbon_intensity(model_excel_db)
fuel_engine_mapping = fuel_engine_mapping[
    fuel_engine_mapping.fuel.isin(OPERATING_FUELS.fuel)
]

# input assumptions for each representative ship
rep_ships = pd.read_pickle(f"{INPUT_PATH}/rep_ships.pkl")
rep_ships['transport_work_tnm_pa'] = annual_transport_work(rep_ships)
retrofit_cost = pd.read_pickle(f"{INPUT_PATH}/retrofit_cost_rep_ships.pkl.gz")
newbuild_premium = pd.read_pickle(f"{INPUT_PATH}/newbuild_premium_rep_ships.pkl")

# # input assumptions for each existing ship
unique_imo_ships = pd.read_pickle(f"{OUTPUT_PATH}/unique_imo_ships.pkl")
unique_imo_ships['transport_work_tnm_pa'] = annual_transport_work(unique_imo_ships)
retrofit_cost_imo_nb = pd.read_pickle(f"{OUTPUT_PATH}/retrofit_cost_unique_imo_ships.pkl.gz")
newbuild_premium_imo_nb = pd.read_pickle(f"{OUTPUT_PATH}/newbuild_premium_unique_imo_ships.pkl")

# common assumptions btw the case study and the full fleet run
start_time = 2025
min_age_for_retrofit = 0

baseline = {
    "generation": start_time,
    "capex_assumption": "Fricaudet (2024)",
    "retrofit_scenario": "Baseline",
    "foresight": 45,
    "discount": 0.08,
    "policy_scenario": "No NZF is a possibility", 
    "efficiency_improvement": "0.0% efficiency impr.",
    "risk_aversion": 0.5,
    "ccs_rate_sc": "Mid",
    "distribution": "uniform",
    "uncertainty_clearing_time_step": 5,
    "retrofit_delay": 0
}


In [ ]:
baseline = {
    "generation": start_time,
    "capex_assumption": "Fricaudet (2024)",
    "retrofit_scenario": "Baseline",
    "foresight": 45,
    "discount": 0.08,
    "policy_scenario": "No NZF is a possibility", 
    "efficiency_improvement": "0.0% efficiency impr.",
    "risk_aversion": 0.5,
    "ccs_rate_sc": "Mid",
    "distribution": "uniform",
    "uncertainty_clearing_time_step": 5,
    "retrofit_delay": 0
}


## Ship case study

### Creating the scenario subsets for sensitivity analysis

In [ ]:
sensitivity_vars = [
    "policy_scenario",
    "foresight",
    "discount",
    "risk_aversion",
    "efficiency_improvement",
    "generation",
    "retrofit_scenario",
    "ccs_rate_sc",
    "distribution",
    "uncertainty_clearing_time_step",
    "retrofit_delay",
    "capex_assumption",
]

# -------------------------
# USER INPUT / BASELINE
# -------------------------
ships = rep_ships[
    (rep_ships.generation.isin([start_time]))
    & (rep_ships.imo_type == "Bulk carrier")
    & (rep_ships.Size_ID == 5)
]

if len(sensitivity_vars) == 0:
    conduct_checks = True
    log = True
    compute_history = True
else:
    conduct_checks = False
    log = False
    compute_history = False

# -------------------------
# DEFINE AVAILABLE VALUES FOR EACH VARIABLE
# (use the same sets you were looping over)
# -------------------------
value_space = {
    "policy_scenario": [
        "No NZF is a possibility",
        "NZF and rewards adopted",
        "IMO Strategy",
    ],
    "discount": [
        0, 
        0.08, 
        0.12
    ],
    "retrofit_scenario": [
        "No retrofit available", 
        "Baseline", 
        "75% of the baseline"
    ],
    "efficiency_improvement": ["0% efficiency impr."],
    "generation": [
        start_time,
    ],
    "risk_aversion": [0, 0.5, 1],
    "ccs_rate_sc": [
        # "Low",
        "Mid",
        # "High"
    ],
    "foresight": [45],
    "distribution":[
        "corners",
        "uniform",
        "triangular"
    ],
    "uncertainty_clearing_time_step": [3, 5, 10],
    "retrofit_delay": [0, 2],
    "capex_assumption": ["Fricaudet (2024)", "IMO's CIA"]
}

# -------------------------
# BUILD THE SCENARIO LIST TO RUN
# baseline + one-variable-at-a-time variations
# -------------------------
runs = [baseline.copy()]  # baseline run

for var in sensitivity_vars:
    if var not in value_space:
        continue
    for val in value_space[var]:
        if val == baseline[var]:
            continue
        sc = baseline.copy()
        sc[var] = val
        runs.append(sc)

# de-duplicate (in case two definitions collide)
runs = [dict(t) for t in {tuple(sorted(d.items())) for d in runs}]

print(f"Number of model runs: {len(runs)}")

# -------------------------
# SET UP INPUTS
# -------------------------
full_results = []
history_full = []
start_time00 = time.time()

segment_size_ids = ships.segment_size_id.unique()
ship_ids = ships.ship_id.unique()

assert operating_cost["# duplicates"].isna().sum() == 0, "NaNs in # duplicates"
pd.DataFrame(runs)

### Run the scenarios

In [ ]:
# -------------------------
# RUN THE REQUIRED SCENARIOS
# -------------------------
for sc in runs:
    # Parse baseline-style fields into what the model needs
    policy_label = sc["policy_scenario"]
    discount_i = sc["discount"]
    retrofit_scenario = sc["retrofit_scenario"]
    risk_aversion = sc["risk_aversion"]
    capture_rate = sc["ccs_rate_sc"]
    foresight = sc["foresight"]
    eff_str = sc["efficiency_improvement"]
    efficiency_improvement = (
        0.0 if eff_str.startswith("0") else float(eff_str.replace("%", "")) / 100.0
    )
    distribution = sc['distribution']
    uncertainty_clearing_time_step = sc['uncertainty_clearing_time_step']
    retrofit_delay = sc['retrofit_delay']
    capex_assumption = sc['capex_assumption']
    assert retrofit_delay < uncertainty_clearing_time_step, (
        f"retrofit_delay ({retrofit_delay}) must be smaller than uncertainty_clearing_time_step "
        f"({uncertainty_clearing_time_step}), otherwise consecutive retrofit-eligibility windows overlap"
    )

    policy_mask = operating_cost["policy_scenario"] == policy_label
    distribution_mask = operating_cost["distribution"] == distribution
    ccs_mask = operating_cost["ccs_rate_sc"] == capture_rate

    mask_op_cost = policy_mask & distribution_mask & ccs_mask
    assert mask_op_cost.any(), f"Empty op_cost for {policy_label=}, {capture_rate=}, {distribution=}"

    delay = int(sc["generation"] - start_time)

    ships_temp = ships.copy()
    ships_temp["generation"] += delay
    ships_temp["scrap_year"] += delay
    current_year = start_time + delay
    ships_temp["cons_pa"] *= 1 - efficiency_improvement
    
    retrofit_sub = retrofit_cost[
        (retrofit_cost.scenario == capex_assumption)
        & (retrofit_cost.segment_size_id.isin(segment_size_ids))
    ].copy()
    retrofit_sub['scenario'] = capex_assumption
    newbuild_sub = newbuild_premium[
        (newbuild_premium.scenario == capex_assumption)
        & (newbuild_premium.ship_id.isin(ship_ids))
    ].copy()
    newbuild_sub['scenario'] = capex_assumption

    if retrofit_scenario == "No retrofit available":
        # if retrofit is impossible, just assume a very high retrofit cost
        retrofit_sub['retrofit_cost'] = pow(10,12)
    elif retrofit_scenario == "75% of the baseline":
        retrofit_sub['retrofit_cost'] *= 0.75


    UNCERTAINTY_CLEARING_YEARS = set(range(current_year + uncertainty_clearing_time_step, 2100, uncertainty_clearing_time_step))
    print(f"Running results for policy_label = {policy_label}, discount = {discount_i}, retrofit_scenario = {retrofit_scenario}, risk_aversion = {risk_aversion}, capture_rate = {capture_rate}, foresight = {foresight}, eff_str = {eff_str}, distribution = {distribution}, uncertainty_clearing_time_step = {uncertainty_clearing_time_step}, retrofit_delay = {retrofit_delay}, ccapex_assumption = {capex_assumption}")
    results_df, history_df = compute_ship_option_values(
        operating_cost[mask_op_cost],
        ships_temp,
        retrofit_sub,
        newbuild_sub,
        drop_in_fuels,
        engines_design_mapping,
        discount_i,
        "lsfo",
        min_age_for_retrofit,
        retrofit_delay,
        UNCERTAINTY_CLEARING_YEARS,
        current_year,
        foresight,
        risk_aversion,
        compute_history,
        conduct_checks,
        log,
        df_engine_efficiency_ref=df_engine_efficiency_ref,
    )

    results_df["foresight"] = foresight
    results_df["retrofit_scenario"] = retrofit_scenario
    results_df["discount"] = discount_i
    results_df["policy_scenario"] = policy_label
    results_df["efficiency_improvement"] = (
        f"{efficiency_improvement*100}% efficiency impr."
    )
    results_df["ccs_rate_sc"] = capture_rate
    results_df["generation"] = start_time + delay
    results_df["risk_aversion"] = risk_aversion
    results_df["distribution"] = distribution
    results_df['retrofit_delay'] = retrofit_delay
    results_df["uncertainty_clearing_time_step"] = uncertainty_clearing_time_step
    results_df["capex_assumption"] = capex_assumption
    results_df["retrofit_scenario"] = retrofit_scenario
    full_results.append(results_df)

    if compute_history:
        history_df["foresight"] = foresight
        history_df["discount"] = discount_i
        history_df["retrofit_scenario"] = retrofit_scenario
        history_df["policy_scenario"] = policy_label
        history_df["ccs_rate_sc"] = capture_rate
        history_df["efficiency_improvement"] = (
            f"{efficiency_improvement*100}% efficiency impr."
        )
        history_df["generation"] = start_time + delay
        history_df["risk_aversion"] = risk_aversion
        history_df["distribution"] = distribution
        history_df['retrofit_delay'] = retrofit_delay
        history_df["uncertainty_clearing_time_step"] = uncertainty_clearing_time_step
        history_df["capex_assumption"] = capex_assumption
        history_df["capex_assumption"] = capex_assumption

        history_full.append(history_df)

end_time = time.time()
elapsed_time = (end_time - start_time00) / 60
print(f"Time spent: {elapsed_time:.1f} minutes")

full_results = pd.concat(full_results, ignore_index=True)
history_full = (
    pd.concat(history_full, ignore_index=True) if compute_history else pd.DataFrame()
)

# Merge for convenience
results_comparison = full_results.merge(
    rep_ships[["imo_type", "Size_ID", "ship_id", "size"]],
    on="ship_id",
    how="left",
)

results_comparison.to_csv(f"{OUTPUT_PATH}/scores_results_rep_ships_{n_scenarios}_{len(runs)}.csv")

history_full.to_pickle(
    f"{OUTPUT_PATH}/history_full_rep_ships_{n_scenarios}_{len(runs)}.pkl.gz",
    compression="gzip",
)

In [ ]:
# paste into the figures.xls file
results_comparison

## Full fleet

In [ ]:
# user_input
ships = unique_imo_ships[
    (unique_imo_ships.scrap_year > start_time)
]

conduct_checks = False
log = False
compute_history = False
risk_aversion = 0.5
current_year = start_time
capex_assumption = "Fricaudet (2024)"
retrofit_scenario = "Baseline"
foresight = 45
policy_label = "No NZF is a possibility"
capture_rate = "Mid"
retrofit_delay = 0

# uncertainty resolution timing: policy/market review years, every
# uncertainty_clearing_time_step years from 2025 to 2050 -- fixed calendar years, not tied to any
# one ship's generation/current_year, since this set is shared across every ship and
# scenario evaluated downstream (Ship case study, Full fleet). At each of these years,
# run_dynamic_programming pools min_cost across all scenarios (weighted by
# `# duplicates`) instead of letting each scenario diverge independently.
uncertainty_clearing_time_step = 5

# prep inputs
# reorder so that conventioanl appear at the top
order = ["conventional"]
ships["fuel_design_cat"] = pd.Categorical(
    ships["initial_fuel_design"],
    categories=order + [c for c in ships["fuel_design"].unique() if c not in order],
    ordered=True,
)
ships = ships.sort_values("fuel_design_cat")

full_results = pd.DataFrame()
start_time00 = time.time()
segment_size_ids = ships.segment_size_id.unique()
ship_ids = ships.ship_id.unique()

op_cost = operating_cost[
    (operating_cost.policy_scenario == policy_label)
    &(operating_cost.distribution == distribution)
    &(operating_cost.ccs_rate_sc == capture_rate)
]
assert operating_cost["# duplicates"].isna().sum() == 0, "NaNs in # duplicates"
UNCERTAINTY_CLEARING_YEARS = set(range(current_year + uncertainty_clearing_time_step, 2100, uncertainty_clearing_time_step))

# Run one for each ship
results_df, history_df = compute_ship_option_values(
    op_cost,
    ships,
    retrofit_cost_imo_nb[
        (retrofit_cost_imo_nb.scenario == capex_assumption)
        & (retrofit_cost_imo_nb.segment_size_id.isin(segment_size_ids))
    ],
    newbuild_premium_imo_nb[
        (newbuild_premium_imo_nb.scenario == capex_assumption)
        & (newbuild_premium_imo_nb.ship_id.isin(ship_ids))
    ],
    drop_in_fuels,
    engines_design_mapping,
    DISCOUNT,
    "lsfo",
    min_age_for_retrofit,
    retrofit_delay,
    UNCERTAINTY_CLEARING_YEARS,
    current_year,
    foresight,
    risk_aversion,
    compute_history,  # compute history
    conduct_checks,  # checks
    log,  # DEBUG
    df_engine_efficiency_ref=df_engine_efficiency_ref,
)

# add some scenario definition for reference
results_df["foresight"] = foresight
results_df["retrofit_scenario"] = retrofit_scenario
results_df["discount"] = DISCOUNT
results_df["policy_scenario"] = policy_label
results_df["ccs_rate_sc"] = capture_rate
results_df["risk_aversion"] = risk_aversion
results_df["distribution"] = distribution
results_df["capex_assumption"] = capex_assumption

end_time = time.time()
elapsed_time = (end_time - start_time00) / 60
print(f"Time spent: {elapsed_time:.4f} minutes")

# cleaning and savings
results_comparison = (
    results_df.reset_index(drop=True)
    .drop_duplicates()
    .merge(
        unique_imo_ships[
            [
                "imo_type",
                "Size_ID",
                "ship_id",
                "size",
                "tj_per_ship_pa",
                "tc_usd_per_day",
                "mcr",
                "dwt",
                "engine_type",
                "Maximum Speed (knots)",
                "speed_exponent",
                "eiv_est"
            ]
        ],
        on="ship_id",
        how="left",
    )
)

results_comparison[results_comparison.risk_indicator_over_tc.notna()].to_csv(f"{OUTPUT_PATH}/scores_results_unique_ships_{n_scenarios}.csv")

print("NA results:")
results_comparison[results_comparison.risk_indicator_over_tc.isna()]

# Plotting results for individual IMO numbers

In [ ]:
n_scenarios = 2500
results_comparison = pd.read_csv(
    f"{OUTPUT_PATH}/scores_results_unique_ships_{n_scenarios}.csv"
).reset_index()

results_comparison = results_comparison.drop_duplicates(
    subset=["ship_id", "policy_scenario", "retrofit_scenario", "foresight"],
    keep=False,
)
len(results_comparison)

In [ ]:
plt.close()
sns.set(style="whitegrid")

df = results_comparison[
    (results_comparison.policy_scenario == "No NZF is a possibility")
    &(results_comparison.speed_exponent == 3)
].copy()
df["Size_ID"] = df["Size_ID"].astype(str)

plt.figure(figsize=(12, 5))

ax = sns.boxplot(
    data=df.sort_values(by=["imo_type", "Size_ID"]),
    x="imo_type",
    y="risk_indicator_over_tc",
    hue="Size_ID",
    palette="tab10",
    dodge=True,
)

ax.set_xlabel("")
ax.set_ylabel("Transition risk score (supply-side)")
ax.legend(
    title="Size bin (1 = small)", 
    loc="upper left", 
    # frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
plt.close()
sns.set(style="whitegrid")

# --- Create 5-year generation bins ---
bin_width = 5
gen_min = (df["generation"] // bin_width) * bin_width
gen_max = gen_min + (bin_width - 1)

df["generation_bin"] = gen_min.astype(str) + "–" + gen_max.astype(str)
bin_order = df.sort_values(by = "generation")["generation_bin"].unique()

plt.figure(figsize=(12, 5))

ax = sns.boxplot(
    data=df.sort_values(by=["imo_type", "generation_bin"]),
    x="imo_type",
    y="risk_indicator_over_tc",
    hue="generation_bin",
    hue_order=bin_order,
    palette="tab10",
    dodge=True,
)

ax.set_xlabel("")
ax.set_ylabel("Transition risk score (supply-side)")
# ax.set_ylim(-0.1,0.4)
ax.legend(
    title="Build year",
    loc="upper left",
    # frameon=False,
)

plt.tight_layout()
plt.show()

# Plotting results - comparison of designs

## Load files

In [ ]:
nb_sensitivities = 14

# results_files
history_full = pd.read_pickle(
    f"{OUTPUT_PATH}/history_full_rep_ships_{n_scenarios}_{nb_sensitivities}.pkl.gz",
    compression="gzip",
)
results_comparison = pd.read_csv(
    f"{OUTPUT_PATH}/scores_results_rep_ships_{n_scenarios}_{nb_sensitivities}.csv"
).reset_index()

# # ships files
rep_ships = pd.read_pickle(f"{INPUT_PATH}/ships/rep_ships.pkl")

intermediate results
drop_in_fuels = pd.read_pickle(f"{OUTPUT_PATH}/drop_in_fuels.pkl")
operating_cost = pd.read_pickle(
    f"{OUTPUT_PATH}/operating_cost_{n_scenarios}_ccs_{ccs}.pkl.gz", compression="gzip"
)

## Sensitivity analysis

In [ ]:
columns_clean = {
    "capex_assumption": "CAPEX input assumptions",
    "discount": "discount factor",
    "generation": "building year",
    "foresight": "foresight (years)",
    "policy_scenario": "policy scenario",
    "efficiency_improvement": "efficiency compared to av.",
    "risk_aversion": "risk aversion",
    "min_year_retrofit": "retrofit availability",
    "ccs_rate_sc": "CCS capture rate",
    "distribution": "random distribution of variables",
    "uncertainty_clearing_time_step": "uncertainty clearing time step (years)",
    "retrofit_delay": "retrofit delay (years)",
    "retrofit_scenario": "retrofit scenario"
}

sensitivity_vars = [
    # "policy_scenario",
    "foresight",
    "discount",
    "risk_aversion",
    # "efficiency_improvement",
    # "generation",
    "capex_assumption",
    # "retrofit_scenario",
    "retrofit_delay",
    # "ccs_rate_sc",
    "distribution",
    "uncertainty_clearing_time_step",
]


def filter_baseline_except(df, baseline, vary_var):
    mask = pd.Series(True, index=df.index)
    for var, val in baseline.items():
        if var != vary_var:
            mask &= df[var] == val
    return df.loc[mask]

df_to_plot = results_comparison[
    (results_comparison.imo_type == "Bulk carrier") & (results_comparison.Size_ID == 5)
]
df_to_plot.uncertainty_clearing_time_step = df_to_plot.uncertainty_clearing_time_step.astype(int)

In [ ]:
plt.close("all")

def wrap_label(s):
    return (
        str(s)
        .replace(" after ", " after\n")
        .replace("based ", "based\n")
        .replace("adopted ", "\nadopted")
        .replace("possibility ", "possibility\n")
        .replace("(", "\n(")
        .replace("IMO's CIA", "Interpolated\n based on the IMO's CIA - low")
    )

low_high_vars = {
    "discount": ("Low", "High"),
    "risk_aversion": ("Low", "High"),
}

base_color_map = {
    "conventional": "black",
    "lng": "#1f77b4",  # blue
    "methanol": "#ff7f0e",  # orange
    "ammonia": "#2ca02c",  # green
}

addon_color_map = {
    "conventional": "0.6",  # light grey
    "lng": "#9ecae1",  # light blue
    "methanol": "#fdd0a2",  # light orange
    "ammonia": "#a1d99b",  # light green
}

def get_base_tech(label):
    label = label.lower()
    if "lng" in label:
        return "lng"
    if "methanol" in label:
        return "methanol"
    if "ammonia" in label:
        return "ammonia"
    return "conventional"


ccs_order = ["Low", "Mid", "High"]

# optional: label cleaning once
df_to_plot = clean_labels_for_plotting(df_to_plot, ["initial_fuel_design"])

vars_to_plot = [v for v in sensitivity_vars if df_to_plot[v].nunique() >= 2]
ncols = 2
nrows = max(1, int(np.ceil(len(vars_to_plot) / ncols)))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), sharey=True)
axes = np.atleast_1d(axes).flatten()

for ax, var in zip(axes, vars_to_plot):
    df_var = filter_baseline_except(df_to_plot, baseline, var)
    df_var = clean_labels_for_plotting(df_var, [var])

    # pivot wide: rows are var values, columns are fuels
    wide = df_var.pivot_table(
        index=var,
        columns="initial_fuel_design",
        values="risk_indicator_over_tc",
        aggfunc="mean",
    ).sort_index()
    # Reindex according to value_space ordering (convert both to string for comparison)
    if var in value_space:
        order = [str(v) for v in value_space[var]]
        var_order = [v for v in order if v in wide.index.values]
        if len(var_order) == len(wide):
            wide = wide.reindex(var_order)
        else:
            print(f"Error in ordering for {var}")

    # plot ALL fuels at once: markers only, no lines
    wide.plot(
        ax=ax,
        marker="o",
        legend=False,  # global legend later
    )
    for line in ax.get_lines():
        label = line.get_label()
        base = get_base_tech(label)

        # baseline (conventional, no add-ons)
        if label.lower() == "conventional":
            line.set_linestyle("--")
            line.set_linewidth(2.5)
            line.set_color(base_color_map[base])

        # add-ons (e.g. + CCS, + sail)
        elif "+ CCS" in label:
            line.set_color(addon_color_map[base])
            line.set_linewidth(1.2)

        # core technologies
        elif "rigid sail" in label:
            line.set_linewidth(1.2)

        else:
            line.set_color(base_color_map[base])
            line.set_linewidth(2.0)

    # baseline tick label
    baseline_value = baseline.get(var, None)
    xtick_labels = []
    for i, v in enumerate(wide.index):
        label = wrap_label(v)

        # mark baseline
        if str(v) == str(baseline_value):
            label = wrap_label(f"{v} – baseline")

        xtick_labels.append(label)

    # add Low / High labels for selected variables
    if var in low_high_vars and len(xtick_labels) >= 2:
        low_label, high_label = low_high_vars[var]

        # prepend "Low" to first tick (unless it is baseline)
        if "baseline" not in xtick_labels[0].lower():
            xtick_labels[0] = f"{xtick_labels[0]} - {low_label}"

        # append "High" to last tick (unless it is baseline)
        if "baseline" not in xtick_labels[-1].lower():
            xtick_labels[-1] = f"{xtick_labels[-1]} - {high_label}"

    ax.set_title(f"Sensitivity to {columns_clean.get(var, var)}")
    ax.set_xlabel("")
    ax.set_ylabel("Transition risk score")
    ax.set_xticks(np.arange(len(wide.index)))
    ax.set_xticklabels(xtick_labels, rotation=0)
    ax.grid(True, alpha=0.3)
    ax.set_xticklabels(xtick_labels)

# remove unused axes
for k in range(len(vars_to_plot), len(axes)):
    fig.delaxes(axes[k])

# one legend for the whole figure (taken from the first plotted axis)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    ncol=1,
    loc="upper center",
    bbox_to_anchor=(.62, .8),
    frameon=False,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Tracking of fuel use

In [ ]:
# user inputs
GEN_TO_PLOT = start_time
GEN_TO_PLOT = 2025
imo_type = "Bulk carrier"
Size_ID = 5
RERUN_OP_CUBE = False

# get the op_cube
scenario_ids = operating_cost["scenario_id"].unique()
years_all = sorted(operating_cost["year"].unique())
designs = drop_in_fuels["fuel_design"].unique().tolist()
engine_types = operating_cost.engine_type.unique()

if RERUN_OP_CUBE:
    _, op_pivot, _ = create_op_cube(
        years_all,
        designs,
        engine_types,
        scenario_ids,
        REG_REG_CLASSES,
        operating_cost,
        drop_in_fuels,
        engines_design_mapping,
        False,
    )
    del _
    gc.collect()

df = clean_results_for_plotting(
    results_comparison,
    history_full,
    op_pivot,
    drop_in_fuels,
    rep_ships,
    designs,
    variables_filter=[
        "policy_scenario",
        "foresight",
        "discount",
        "risk_aversion",
        "retrofit_scenario",
        "uncertainty_clearing_time_step",
        "retrofit_delay",
        "distribution",
        "capex_assumption"
    ],
    values_filter=[
        baseline["policy_scenario"],
        baseline["foresight"],
        baseline["discount"],
        baseline["risk_aversion"],
        baseline["retrofit_scenario"],
        baseline["uncertainty_clearing_time_step"],
        baseline["retrofit_delay"],
        baseline["distribution"],
        baseline["capex_assumption"]
    ],
    imo_type=imo_type,
    size_id=Size_ID,
    current_year=GEN_TO_PLOT,
)
df.sample(3)


In [ ]:

def lighten_color(color, amount=0.6):
    """
    Lighten a color by blending it with white.
    amount=0 -> original; amount=1 -> white
    """
    r, g, b = mcolors.to_rgb(color)
    return (
        (1 - amount) * r + amount,
        (1 - amount) * g + amount,
        (1 - amount) * b + amount,
    )

def addon_type(label: str) -> str | None:
    s = str(label).lower()
    if "ccs" in s:
        return "ccs"
    if ("rigid sail" in s) or ("sail" in s) or ("wind" in s):
        return "sail"
    if "+" in s:
        return "other"
    return None

def base_fuel_family(label: str) -> str:
    """
    Map a fuel_end label to a base fuel family used for consistent colours.
    Adjust rules to match your exact naming conventions.
    """
    s = str(label).lower()

    # strip common add-on patterns after plus
    if "+" in s:
        s = s.split("+", 1)[0].strip()

    # family rules (edit to match your labels)
    if "lng" in s:
        return "LNG"
    if "methanol" in s:
        return "Methanol"
    if "ammonia" in s:
        return "Ammonia"
    if (
        "hfo" in s
        or "vlsfo" in s
        or "lsfo" in s
        or "mgo" in s
        or "diesel" in s
        or "conventional" in s
        or "fuel oil" in s
    ):
        return "Conventional"
    if "hydrogen" in s:
        return "Hydrogen"
    # fallback: use cleaned string as its own family
    return s.strip().title()


# Choose base colours for families (edit as you like)
family_colors = {
    "Conventional": "black",
    "LNG": "#1f77b4",  # blue
    "Ammonia": "#ff7f0e",  # orange
    "Methanol": "#2ca02c",  # green
    "Hydrogen": "#9467bd",  # purple
}

addon_style = {
    "ccs": {"lighten": 0.3, "hatch": None, "alpha": 1.0},
    "sail": {"lighten": 0.80, "hatch": None, "alpha": 1.0},  # hatch differentiates
    "other": {"lighten": 0.60, "hatch": "..", "alpha": 1.0},
}

In [ ]:
# plot fuel designs

REG_REG_CLASS_TO_PLOT = "noneu_to_noneu"
engine_types = ["LNG Diesel 2024+", "All"]

df_to_plot = df.copy()
df_to_plot = clean_labels_for_plotting(df_to_plot, ["initial_fuel_design", "fuel_end"])

all_fuels = sorted(df_to_plot["fuel_end"].unique())

# Ensure numeric
df_to_plot["# duplicates"] = pd.to_numeric(df_to_plot["# duplicates"])

fuel_designs = df_to_plot["initial_fuel_design"].unique()

n_rows = int((len(fuel_designs) + 1) / 2)
n_cols = 2

plt.close()
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(6 * n_cols, 3 * n_rows), sharex=True, sharey=True
)

axes = np.atleast_1d(axes).flatten()
# ------------------------------------------------------------
# Palette
# ------------------------------------------------------------

# Assign final colours: base = family colour; add-on = lightened family colour
fuel_to_color = {}
fuel_to_hatch = {}
fuel_to_alpha = {}

for fuel in all_fuels:
    fam = base_fuel_family(fuel)
    base_col = family_colors[fam]

    a = addon_type(fuel)
    if a is None:
        fuel_to_color[fuel] = base_col
        fuel_to_hatch[fuel] = None
        fuel_to_alpha[fuel] = 1.0
    else:
        st = addon_style.get(a, addon_style["other"])
        fuel_to_color[fuel] = lighten_color(base_col, amount=st["lighten"])
        fuel_to_hatch[fuel] = st["hatch"]
        fuel_to_alpha[fuel] = st["alpha"]

global_handles = {}

# ------------------------------------------------------------
# Plot loop
# ------------------------------------------------------------
for ax, fuel_design in zip(axes, fuel_designs):

    sub = df_to_plot[
        (df_to_plot["initial_fuel_design"] == fuel_design)
        & (df_to_plot["reg_reg_class"] == REG_REG_CLASS_TO_PLOT)
        & (df_to_plot.engine_type_end.isin(engine_types))
        & (df_to_plot.scenario_id.isin(scenario_ids))
    ]

    grouped = (
        sub.groupby(["year", "fuel_end"])["# duplicates"].sum().unstack(fill_value=0)
    )

    grouped = grouped.sort_index()

    years = grouped.index
    n_years = len(years)

    bottom = pd.Series([0] * n_years, index=years)

    for fuel in all_fuels:

        if fuel in grouped.columns:
            values = grouped[fuel]
        else:
            values = pd.Series([0] * n_years, index=years)

        bars = ax.bar(
            years, values.values, bottom=bottom.values, color=fuel_to_color[fuel]
        )

        if fuel not in global_handles:
            global_handles[fuel] = bars

        bottom += values

    ax.set_title(f"{fuel_design}")
    ax.set_xlabel("")
    ax.set_ylabel("# scenarios")

# ------------------------------------------------------------
# Legend (RIGHT SIDE, VERTICAL)
# ------------------------------------------------------------
fig.legend(
    handles=list(global_handles.values()),
    labels=list(global_handles.keys()),
    loc="center left",
    bbox_to_anchor=(0, -0.12),
    frameon=True,
)

plt.tight_layout(rect=[0, 0, 0.88, 1])  # leave space on right for legend
plt.show()

In [ ]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
REG_REG_CLASS_TO_PLOT = "noneu_to_noneu"
engine_types = ["LNG Diesel 2024+", "All"]

# ------------------------------------------------------------
# Make a copy of the dataframe to work with
# ------------------------------------------------------------
df_plot = df[
    (df["reg_reg_class"] == REG_REG_CLASS_TO_PLOT)
    & (df.engine_type_end.isin(engine_types))
]

# Ensure numeric
df_plot["# duplicates"] = pd.to_numeric(df_plot["# duplicates"])

# Remove " + rigid sail" from fuel names in the copy only
df_plot["fuel"] = df_plot["fuel"].str.replace(r"_\+_rigid_sail$", "", regex=True)
df_plot["fuel"] = df_plot["fuel"].str.replace(r"_\+_ccs$", "", regex=True)

df_plot = clean_labels_for_plotting(
    df_plot, ["initial_fuel_design", "fuel_end", "fuel"]
)

fuel_designs = df_plot["initial_fuel_design"].unique()
n_rows = int((len(fuel_designs) + 1) / 2)
n_cols = 2

plt.close()
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(6 * n_cols, 3 * n_rows), sharex=True, sharey=True
)
axes = np.atleast_1d(axes).flatten()

# ------------------------------------------------------------
# Palette
# ------------------------------------------------------------
all_fuels = sorted(df_plot["fuel"].unique())

palette1 = sns.color_palette("tab20")
combined_palette = palette1 + [(0, 0, 0), (0.5, 0.5, 0.5),  (0.7, 0.7, 0.7),  (0.9, 0.9, 0.9),  (0.2, 0.2, .2),  (0.4, 0.4,.4)]  # total 32+ colors

fuel_to_color = {fuel: combined_palette[i] for i, fuel in enumerate(all_fuels)}

global_handles = {}

# ------------------------------------------------------------
# Plot loop
# ------------------------------------------------------------
for ax, fuel_design in zip(axes, fuel_designs):

    subsub = df_plot[
        (df_plot["initial_fuel_design"] == fuel_design)
        & (df_plot.scenario_id.isin(scenario_ids))
    ]

    grouped = (
        subsub.groupby(["year", "fuel"])["# duplicates"].sum().unstack(fill_value=0)
    )

    grouped = grouped.sort_index()
    years = grouped.index
    n_years = len(years)
    bottom = pd.Series([0] * n_years, index=years)

    for fuel in all_fuels:
        if fuel in grouped.columns:
            values = grouped[fuel]
        else:
            values = pd.Series([0] * n_years, index=years)

        bars = ax.bar(
            years, values.values, bottom=bottom.values, color=fuel_to_color[fuel]
        )

        if fuel not in global_handles:
            global_handles[fuel] = bars

        bottom += values

    ax.set_title(f"{fuel_design}")
    ax.set_xlabel("")
    ax.set_ylabel("# scenarios")

# ------------------------------------------------------------
# Legend (RIGHT SIDE, VERTICAL)
# ------------------------------------------------------------
fig.legend(
    handles=list(global_handles.values()),
    labels=list(global_handles.keys()),
    loc="center left",
    bbox_to_anchor=(0, -0.2),
    frameon=True,
)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()